In [ ]:
from transformers import pipeline
from datasets import load_dataset
import matplotlib.pyplot as plt
import pandas as pd

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch
nltk.download('punkt')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
model_id = "google/pegasus-cnn_dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)
dataset = load_dataset("ccdv/arxiv-summarization")

In [ ]:
from datasets import DatasetDict

dataset = DatasetDict({
    "train": dataset["train"].select(range(500)),
    "validation": dataset["validation"].select(range(50)),
    "test": dataset["test"].select(range(50))
}) #not training the full dataset to save time and resources

In [ ]:
def createTokens(examples):

    input_encoding = tokenizer(
        examples["article"],
        truncation=True,
        max_length=256 #saving compputation time by limiting the input length to 512 tokens
    )
    target_encoding = tokenizer(
        text_target=examples["abstract"],
        truncation=True,
        max_length=64
    )

    return {
        "input_ids": input_encoding['input_ids'],
        "attention_mask": input_encoding['attention_mask'],
        "labels": target_encoding['input_ids']
    }
    
dataset_tokenized = dataset.map(createTokens,batched=True)

In [ ]:
dataset_tokenized

In [ ]:
from transformers import DataCollatorForSeq2Seq
sequence_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",

    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,

    fp16=True,
    gradient_checkpointing=True,

    predict_with_generate=True,
    generation_max_length=128,

    eval_strategy="no",
    report_to="none"
)


In [ ]:
import evaluate

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    labels = [
        [
            token if token != -100 else tokenizer.pad_token_id
            for token in label
        ]
        for label in labels
    ]

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )
    
    decoded_preds = [
        pred.strip() for pred in decoded_preds
    ]

    decoded_labels = [
        label.strip() for label in decoded_labels
    ]

    rouge = evaluate.load("rouge")
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return {
        key: round(value, 4)
        for key, value in result.items()
    }

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    data_collator=sequence_data_collator,
    train_dataset=dataset_tokenized["test"], #doing on test to save time and resources, but should be done on train dataset for real training
    eval_dataset=dataset_tokenized["validation"],
    compute_metrics=compute_metrics
)
model.gradient_checkpointing_enable()
model.config.use_cache = False

trainer.train()

In [ ]:
trainer.state.log_history

In [ ]:
small_validation = dataset_tokenized["validation"].select(range(50))

results = trainer.evaluate(
    eval_dataset=small_validation
)

print(results)

In [ ]:
model.save_pretrained("pegasus-finetuned-arxiv")
tokenizer.save_pretrained("pegasus-finetuned-arxiv")

In [ ]:
import transformers

print(transformers.__version__)